# Baisc RAG Program

In [26]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
load_dotenv()  # LangChain will automatically pick up the GOOGLE_API_KEY from the environment

# LLM / Model
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite',temperature=0)

embeddings_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001") 

In [1]:
from langchain_community.document_loaders import WebBaseLoader
import bs4

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# Embed amd store
vectorstore = Chroma.from_documents(documents=splits, 
                                    embedding=embeddings_model)

retriever = vectorstore.as_retriever()

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [38]:
# Prompt
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")  # Pre-define prompt used for basic RAG purpose and two input exist in the prompt - 'context', 'question'

# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Result
rag_chain.invoke("What is Task Decomposition?")

'Task decomposition is the process of breaking down a complex task into smaller, more manageable sub-tasks. This approach helps in organizing and executing tasks more efficiently. By dividing a large problem into smaller pieces, it becomes easier to understand, plan, and implement solutions.'

In [41]:
# Spliting the text based on token size
import tiktoken
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=300, chunk_overlap=50)
splits = text_splitter.split_documents(docs)

In [ ]:
#  Cosine Similarity
import numpy as np
def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

query_result = embeddings_model.embed_query(question)
document_result = embeddings_model.embed_query(document)

similarity = cosine_similarity(query_result, document_result)
print("Cosine Similarity:", similarity)



# ---------------------------------------------------------------------

# Input user Query Translation 
- Install all package:->  ! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain
- Standard defined methods: Multi-Query, RAG-FUsion, Decomposition, Step-back, HyDE


# Multi Query
- an advanced strategy where a single user query is automatically rewritten into multiple, distinct queries

**Parallel retrieval:** Each of these generated queries is used to perform a separate search against the vector database.

**Result combination:** The system collects the relevant documents retrieved from all the parallel searches. It then takes the unique union of these documents to create a larger, richer set of contexts. In advanced versions, a re-ranking step might be included to prioritize the most relevant documents.

**Augmented response generation:** The combined, potentially deduplicated documents are passed along with the original user query to a final LLM. This LLM synthesizes the information from the multiple perspectives to generate a more robust and accurate final answer.

In [45]:
# Prompt for multi query:

from langchain.prompts import ChatPromptTemplate

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Original question: {question}
note: only provide queries in output, no other content"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser

generate_queries = (
    prompt_perspectives 
    | llm
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)


# Document processing
from langchain.load import dumps, loads

def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]

In [63]:
# Retrieve
question = "What is task decomposition for LLM agents?"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
# docs = retrieval_chain.invoke({"question":question})
# len(docs)

7

In [65]:
# Final invking the model by passsing the question
from operator import itemgetter
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain, "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

'Task decomposition for LLM agents can be achieved through several methods:\n\n*   **LLM with simple prompting:** This involves using prompts like "Steps for XYZ.\\\\n1." or "What are the subgoals for achieving XYZ?".\n*   **Task-specific instructions:** Providing explicit instructions tailored to the task, such as "Write a story outline." for novel writing.\n*   **Human inputs:** Incorporating human guidance to break down tasks.\n*   **LLM+P (External Classical Planner):** This approach uses an external classical planner with the Planning Domain Definition Language (PDDL) as an interface. The LLM translates the problem into "Problem PDDL", a classical planner generates a PDDL plan based on "Domain PDDL", and the LLM then translates the plan back into natural language. This method outsources the planning step to an external tool, assuming the availability of domain-specific PDDL and a suitable planner.'

# RAG-Fusion
- a technique that leverages a large language model (LLM) to generate multiple, similar questions from a single user query.
- The goal is to broaden the scope of the search, capture different interpretations of the user's intent, and retrieve a more diverse and comprehensive set of documents.

**Perform parallel retrieval:** The system conducts a separate vector search using each of these generated queries against its knowledge base.

**Reciprocal Rank Fusion (RRF):** RAG-Fusion employs an algorithm called Reciprocal Rank Fusion (RRF) to combine and re-rank the retrieved documents from all the parallel searches. This process prioritizes documents that consistently rank highly across multiple queries.

**Synthesize the final response:** The final, re-ranked set of documents is then passed to the LLM, which uses this robust, multi-perspective context to generate a more accurate and comprehensive answer.

In [67]:
from langchain.prompts import ChatPromptTemplate

# RAG-Fusion: Related
template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (5 queries):
note: only provide queries in output, no other content"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser

generate_queries = (
    prompt_rag_fusion 
    | llm
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)


In [69]:
#from langchain.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents 
        and an optional parameter k used in the RRF formula """
    
    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question": question})
len(docs)

9

question = "What is task decomposition for LLM agents?"

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

# Decomposition

- A technique that breaks down a complex user query into smaller, simpler, and more manageable sub-questions. 
- By addressing each sub-question individually, the system can retrieve more precise and targeted information, and then synthesize the results into a comprehensive final answer. 
- This is especially useful for "multi-hop" questions, where the answer requires information from multiple documents.

**Parallel or sequential retrieval:** The RAG system executes separate retrieval searches for each of the sub-questions against its knowledge base. The sub-queries can be run in parallel or sequentially depending on the complexity and dependencies.

**Consolidation and reasoning:** The retrieved documents for each sub-question are gathered and presented to a final LLM. This LLM then uses the combined context to reason about the answer to the original complex query.

**Final response:** The LLM synthesizes the information to generate a cohesive and accurate response to the user.

In [75]:
# Decomposition
template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):
note: only provide queries in output, no other content """
prompt_decomposition = ChatPromptTemplate.from_template(template)

# Chain
generate_queries_decomposition = ( prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n")))

# Run
question = "What are the main components of an LLM-powered autonomous agent system?"
questions = generate_queries_decomposition.invoke({"question":question})

questions

['LLM agent architecture components',
 'Key elements of autonomous AI agents',
 'LLM agent system design principles']

# Decomposition : Answer recursively

In [76]:
# Prompt
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question: 

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

In [79]:
def format_qa_pair(question, answer):
    """Format Q and A pair"""
    
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

q_a_pairs = ""
for q in questions:
    
    rag_chain = (
    {"context": itemgetter("question") | retriever, 
     "question": itemgetter("question"),
     "q_a_pairs": itemgetter("q_a_pairs")} 
    | decomposition_prompt
    | llm
    | StrOutputParser())

    answer = rag_chain.invoke({"question":q,"q_a_pairs":q_a_pairs})
    q_a_pair = format_qa_pair(q,answer)
    q_a_pairs = q_a_pairs + "\n---\n"+  q_a_pair

In [80]:
answer

"The design principles for LLM agent systems revolve around enabling autonomous and goal-oriented behavior. Key elements that underpin these systems include:\n\n*   **Memory:** This is fundamental for an agent's ability to learn, adapt, and maintain context.\n    *   **Short-term memory:** This refers to the agent's in-context learning capabilities, allowing it to draw information from the immediate conversation or prompt.\n    *   **Long-term memory:** This enables the agent to retain and recall information over extended periods. This is typically achieved by using external vector stores and fast retrieval mechanisms.\n\n*   **Tool Use:** Agents are designed to interact with external APIs and resources to access information or perform actions beyond their inherent model weights. This capability allows them to:\n    *   Retrieve up-to-date information from external sources.\n    *   Execute code to perform computations or manipulate data.\n    *   Access proprietary data sources to enr

# Decomposition : Answer individually

In [82]:
# RAG prompt
question = "What are the main components of an LLM-powered autonomous agent system?"
prompt_rag = hub.pull("rlm/rag-prompt")

def retrieve_and_rag(question,prompt_rag,sub_question_generator_chain):
    """RAG on each sub-question"""
    
    # Use our decomposition / 
    sub_questions = sub_question_generator_chain.invoke({"question":question})
    
    # Initialize a list to hold RAG chain results
    rag_results = []
    
    for sub_question in sub_questions:
        
        # Retrieve documents for each sub-question
        retrieved_docs = retriever.get_relevant_documents(sub_question)
        
        # Use retrieved documents and sub-question in RAG chain
        answer = (prompt_rag | llm | StrOutputParser()).invoke({"context": retrieved_docs, "question": sub_question})
        rag_results.append(answer)
    
    return rag_results,sub_questions

# Wrap the retrieval and RAG process in a RunnableLambda for integration into a chain
answers, questions = retrieve_and_rag(question, prompt_rag, generate_queries_decomposition)

C:\Users\white\anaconda3\envs\env_rag\lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(
C:\Users\white\AppData\Local\Temp\ipykernel_10824\3575482236.py:16: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retrieved_docs = retriever.get_relevant_documents(sub_question)


In [83]:
def format_qa_pairs(questions, answers):
    """Format Q and A pairs"""
    
    formatted_string = ""
    for i, (question, answer) in enumerate(zip(questions, answers), start=1):
        formatted_string += f"Question {i}: {question}\nAnswer {i}: {answer}\n\n"
    return formatted_string.strip()

context = format_qa_pairs(questions, answers)

# Prompt
template = """Here is a set of Q+A pairs:

{context}

Use these to synthesize an answer to the question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context":context,"question":question})

'Based on the provided Q&A pairs, the main components of an LLM-powered autonomous agent system are:\n\n*   **Memory:** This includes both short-term memory, often referred to as in-context learning, and long-term memory, which can be implemented using external vector stores.\n*   **Tool Use:** This allows the agent to interact with external APIs, enabling it to access additional information or perform specific capabilities.'

# Step Back

- An advanced strategy where the system uses a large language model (LLM) to first rephrase a user's complex or specific query into a more abstract, high-level "step-back" question. 
- This broader question retrieves general, conceptual information, which is then used alongside the original query to form a more robust and accurate final answer.

### How the Step-Back technique works

**Original query:** The user submits a specific, detailed query.
        Example: "What were the specific features of the first iPhone?"
        
**Generate a step-back query:** The LLM, using a special prompt, is instructed to generate a more abstract, conceptual query from the original one.
        Example: "What are the general design principles and major features of early smartphones?"
        
**Parallel retrieval:** The system performs two separate retrieval tasks in parallel:
        Vector search with the original query: This retrieves documents highly relevant to the specific details of the original question.
        Vector search with the step-back query: This retrieves documents focused on the broader, more foundational concepts.
        
**Consolidate and reason:** Both sets of retrieved documents (the specific and the general) are combined and passed as context to the final LLM, along with the original user query.

In [84]:
# Few Shot Examples
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel’s was born in what country?",
        "output": "what is Jan Sindel’s personal history?",
    },
]
# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        # Few shot examples
        few_shot_prompt,
        # New question
        ("user", "{question}"),
    ]
)

In [86]:
# The RunnableLambda(lambda x: x["question"]) has the same effect as itemgetter("question"). 
# In general, itemgetter is slightly faster and often preferred for this simple use case. 
# However, RunnableLambda with a custom lambda offers greater flexibility for more complex, in-line data manipulation, such as RunnableLambda(lambda x: x["question"].upper())

generate_queries_step_back = prompt | llm | StrOutputParser()
question = "What is task decomposition for LLM agents?"
generate_queries_step_back.invoke({"question": question})


'What are the fundamental components of LLM agents?'

In [89]:
# Response prompt 
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""
response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

chain = (
    {
        # Retrieve context using the normal question
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        # Retrieve context using the step-back question
        "step_back_context": generate_queries_step_back | retriever,
        # Pass on the question
        "question": lambda x: x["question"],
    }
    | response_prompt
    | llm
    | StrOutputParser()
)

chain.invoke({"question": question})

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting.
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 4.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting.
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting.


'Task decomposition for LLM agents refers to the process of breaking down a complex, overarching goal into smaller, more manageable sub-tasks or sub-goals. This allows the agent to tackle the larger objective in a structured and sequential manner.\n\nThere are several ways task decomposition can be achieved:\n\n*   **LLM with Simple Prompting:** The LLM itself can be prompted to perform the decomposition. This might involve using prompts like "Steps for XYZ" or "What are the subgoals for achieving XYZ?".\n*   **Task-Specific Instructions:** Pre-defined, task-specific instructions can guide the decomposition. For example, for the task of writing a novel, an instruction like "Write a story outline" would initiate the decomposition process.\n*   **Human Inputs:** Humans can also provide direct input to guide the decomposition of tasks.\n\nAn alternative approach, known as **LLM+P**, involves outsourcing the long-horizon planning aspect to an external classical planner. This method uses th

# Hypothetical Document Embeddings (HyDE)

- A query translation technique that significantly improves retrieval accuracy, especially for short, vague, or complex queries.

### How HyDE works

**Generate a hypothetical document:** When a user submits a query (e.g., "how to fix a leaky pipe?"), an instruction-following LLM (like GPT-4) generates a detailed, hypothetical document that could potentially answer the query.

**Embed the hypothetical document:** The generated hypothetical document is converted into a vector embedding using an embedding model. The key insight here is that this detailed, hypothetical document is likely to be closer in the vector space to the actual documents containing the answer than the original, often shorter, user query.

**Perform similarity search:** This new embedding is used to search the vector database for real documents that are semantically similar to the hypothetical document.

**Retrieve and generate:** The most relevant documents are retrieved and passed to a generative LLM along with the original user query, enabling it to produce a more grounded and accurate final response.

In [ ]:
# HyDE document generation
template = """Please write a scientific paper passage to answer the question
Question: {question}
Passage:"""
prompt_hyde = ChatPromptTemplate.from_template(template)


generate_docs_for_retrieval = (
    prompt_hyde | llm | StrOutputParser() 
)

# Run
question = "What is task decomposition for LLM agents?"
generate_docs_for_retrieval.invoke({"question":question})

In [92]:
# Retrieve
retrieval_chain = generate_docs_for_retrieval | retriever 
retrieved_docs = retrieval_chain.invoke({"question":question})
retrieved_docs

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.\nAnother quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planning. This approach utilizes the Planning Domain Definition Language (PDDL) as an intermediate interface to describe the planning problem. In this process, LLM (1) translates the problem into “Problem PDDL”, then (2) requests a classical planner to generate a PDDL plan based on an existing “Domain PDDL”, and finally (3) translates the PDDL plan back into natural language. Essentially, the planning step is outsourced to an external tool, assuming the availability of domain-specific PDDL and a suitable planner

In [93]:


# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context":retrieved_docs,"question":question})



'Task decomposition for LLM agents is the process of breaking down large tasks into smaller, manageable subgoals. This enables the agent to handle complex tasks more efficiently. This decomposition can be achieved through:\n\n*   **Simple prompting:** Using prompts like "Steps for XYZ.\\\\n1." or "What are the subgoals for achieving XYZ?".\n*   **Task-specific instructions:** Providing instructions tailored to the task, such as "Write a story outline." for novel writing.\n*   **Human inputs:** Receiving direct input from a human to define the subgoals.'

# ---------------------------------------------------------------------

# Input Query Routing
- directs a user's query to the most suitable data source, retrieval method, or processing pipeline
- required modules:  ! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain youtube-transcript-api pytube

### Methods for implementing query routers
There are several ways to implement the decision-making logic of a query router. 

**LLM Router:** Uses a large language model to classify the query's intent by prompting it with descriptions of the available options. This is flexible and can handle complex or ambiguous queries.

**Semantic Router:** Converts the user's query and a set of example queries for each route into vector embeddings. The router then uses a similarity search to find the closest match and route the query accordingly. This is often more efficient than an LLM-based router.

**Logical or Keyword-Based Router:** Uses predefined rules and keyword filtering to direct the query. While simple to implement, this method can be less robust than LLM or semantic routing as it struggles with variations in phrasing.

# Logical Routing

In [97]:
# Data model
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import Literal

class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""

    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
        ...,
        description="Given a user question choose which datasource would be most relevant for answering their question",
    )

# LLM with function call 
structured_llm = llm.with_structured_output(RouteQuery)

# Prompt 
system = """You are an expert at routing a user question to the appropriate data source.

Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

# Define router 
router = prompt | structured_llm

# ---------------------------------------
question = """Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

result = router.invoke({"question": question})
# result.datasource

In [105]:
def choose_route(result):
    if "python_docs" in result.datasource.lower():
        ### Logic here 
        return "chain for python_docs"
    elif "js_docs" in result.datasource.lower():
        ### Logic here 
        return "chain for js_docs"
    else:
        ### Logic here 
        return "golang_docs"

from langchain_core.runnables import RunnableLambda

full_chain = router | RunnableLambda(choose_route)

full_chain.invoke({"question": question})

'chain for python_docs'

# Semantic Routing

In [109]:
from langchain.utils.math import cosine_similarity
from langchain_core.prompts import PromptTemplate

# Two prompts
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

# Embed prompts
embeddings = embeddings_model
prompt_templates = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_templates)

# Route question to prompt 
def prompt_router(input):
    # Embed question
    query_embedding = embeddings.embed_query(input["query"])
    # Compute similarity
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    most_similar = prompt_templates[similarity.argmax()]
    # Chosen prompt 
    print("Using MATH" if most_similar == math_template else "Using PHYSICS")
    return PromptTemplate.from_template(most_similar)


chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | llm
    | StrOutputParser()
)

print(chain.invoke("What's a black hole"))

Using MATH
Ah, a question about one of the most fascinating and enigmatic objects in the universe! As a mathematician, I love tackling concepts that push the boundaries of our understanding, and black holes certainly do that. Let's break down what a black hole is by looking at its fundamental components and the principles that govern it.

To understand a black hole, we need to consider a few key mathematical and physical concepts:

**1. Gravity and Escape Velocity:**

*   **The Concept:** Gravity is the force of attraction between any two objects with mass. The more massive an object, and the closer you are to it, the stronger its gravitational pull.
*   **Escape Velocity:** Imagine throwing a ball upwards. It will eventually fall back down due to Earth's gravity. If you could throw it fast enough, it would escape Earth's gravitational pull and never return. This minimum speed required to escape a celestial body's gravity is called its **escape velocity**.
*   **Mathematical Relation:*

In [ ]:
# -------------------------------------------------------------------------------

# Query Construction

In [116]:
import datetime
from typing import Literal, Optional, Tuple
from langchain_core.pydantic_v1 import BaseModel, Field

class TutorialSearch(BaseModel):
    """Search over a database of tutorial videos about a software library."""

    content_search: str = Field(
        ...,
        description="Similarity search query applied to video transcripts.",
    )
    title_search: str = Field(
        ...,
        description=(
            "Alternate version of the content search query to apply to video titles. "
            "Should be succinct and only include key words that could be in a video "
            "title."
        ),
    )
    min_view_count: Optional[int] = Field(
        None,
        description="Minimum view count filter, inclusive. Only use if explicitly specified.",
    )
    max_view_count: Optional[int] = Field(
        None,
        description="Maximum view count filter, exclusive. Only use if explicitly specified.",
    )
    earliest_publish_date: Optional[datetime.date] = Field(
        None,
        description="Earliest publish date filter, inclusive. Only use if explicitly specified.",
    )
    latest_publish_date: Optional[datetime.date] = Field(
        None,
        description="Latest publish date filter, exclusive. Only use if explicitly specified.",
    )
    min_length_sec: Optional[int] = Field(
        None,
        description="Minimum video length in seconds, inclusive. Only use if explicitly specified.",
    )
    max_length_sec: Optional[int] = Field(
        None,
        description="Maximum video length in seconds, exclusive. Only use if explicitly specified.",
    )

    def pretty_print(self) -> None:
        for field in self.__fields__:
            if getattr(self, field) is not None and getattr(self, field) != getattr(
                self.__fields__[field], "default", None
            ):
                print(f"{field}: {getattr(self, field)}")



In [111]:
system = """You are an expert at converting user questions into database queries. \
You have access to a database of tutorial videos about a software library for building LLM-powered applications. \
Given a question, return a database query optimized to retrieve the most relevant results.

If there are acronyms or words you are not familiar with, do not try to rephrase them."""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)
structured_llm = llm.with_structured_output(TutorialSearch)
query_analyzer = prompt | structured_llm

In [117]:
query_analyzer.invoke({"question": "rag from scratch"}).pretty_print()


content_search: rag from scratch
title_search: rag from scratch


In [118]:
query_analyzer.invoke({"question": "find a video on AI bubble news"}).pretty_print()


content_search: AI bubble news
title_search: AI bubble news


In [119]:
query_analyzer.invoke(
    {
        "question": "how to use multi-modal models in an agent, only videos under 5 minutes"
    }
).pretty_print()



content_search: multi-modal models in an agent
title_search: multi-modal agent
max_length_sec: 300


In [ ]:
# ---------------------------------------------------------------------------

# Indexing
- It includes Chunk optimization, Multi-representation Indexing, specialized embedding, hierarchical indexing

# Multi-Representation Indexing

In [121]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

loader = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
docs.extend(loader.load())

In [122]:
import uuid

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{doc}")
    | llm
    | StrOutputParser()
)

summaries = chain.batch(docs, {"max_concurrency": 5})



In [126]:
from langchain.storage import InMemoryByteStore
from langchain_chroma import Chroma
#from langchain_community.vectorstores import Chroma
from langchain.retrievers.multi_vector import MultiVectorRetriever

# The vectorstore to use to index the child chunks
vectorstore = Chroma(collection_name="summaries",
                     embedding_function=embeddings_model)

# The storage layer for the parent documents
store = InMemoryByteStore()
id_key = "doc_id"

# The retriever
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)
doc_ids = [str(uuid.uuid4()) for _ in docs]

# Docs linked to summaries
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

# Add
retriever.vectorstore.add_documents(summary_docs)
retriever.docstore.mset(list(zip(doc_ids, docs)))

In [127]:
query = "Memory in agents"
sub_docs = vectorstore.similarity_search(query,k=1)
sub_docs[0]

Document(id='78fea656-b435-4afd-b472-e676b6d38def', metadata={'doc_id': '2542ebae-6c65-44c5-90de-6d4142548036'}, page_content='This document, "LLM Powered Autonomous Agents" by Lilian Weng, provides a comprehensive overview of building autonomous agents where Large Language Models (LLMs) serve as the core controller.\n\nThe article breaks down the architecture of such agents into three key components:\n\n1.  **Planning:** This involves the agent\'s ability to break down complex tasks into smaller, manageable subgoals (task decomposition) and to reflect on its past actions to learn from mistakes and refine its strategy (self-reflection). Techniques like Chain of Thought (CoT), Tree of Thoughts (ToT), ReAct, and Reflexion are discussed as methods for achieving this.\n\n2.  **Memory:** Agents need to store and retrieve information. The article categorizes memory into sensory, short-term (akin to in-context learning), and long-term memory (often implemented with external vector stores and 

In [128]:
retrieved_docs = retriever.get_relevant_documents(query,n_results=1)
retrieved_docs[0].page_content[0:500]

"\n\n\n\n\n\nLLM Powered Autonomous Agents | Lil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n|\n\n\n\n\n\n\nPosts\n\n\n\n\nArchive\n\n\n\n\nSearch\n\n\n\n\nTags\n\n\n\n\nFAQ\n\n\n\n\n\n\n\n\n\n      LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\n \n\n\nTable of Contents\n\n\n\nAgent System Overview\n\nComponent One: Planning\n\nTask Decomposition\n\nSelf-Reflection\n\n\nComponent Two: Memory\n\nTypes of Memory\n\nMaximum Inner Product Search (MIPS)\n\n\nComponent Three:"

# RAPTOR 
- Recursive Abstractive Processing for Tree-Organized Retrieval
- Deep dive video: https://www.youtube.com/watch?v=jbGchdTL7d0
- Python code example :  https://github.com/langchain-ai/langchain/blob/master/cookbook/RAPTOR.ipynb 

# ColBERT
- RAGatouille makes it as simple to use ColBERT.
- ColBERT generates a contextually influenced vector for each token in the passages.
- ColBERT similarly generates vectors for each token in the query.
- Then, the score of each document is the sum of the maximum similarity of each query embedding to any of the document embeddings:
- ! pip install -U ragatouille

# Self RAG:
- Self-RAG is a strategy for RAG that incorporates self-reflection / self-grading on retrieved documents and generations.
- https://github.com/langchain-ai/langgraph/blob/main/examples/rag/langgraph_self_rag.ipynb

In [134]:
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)
doc_splits = text_splitter.split_documents(docs_list)

# Add to vectorDB
vectorstore = Chroma.from_documents(
    documents=doc_splits,
    collection_name="rag-chroma",
    embedding= embeddings_model,
)
retriever = vectorstore.as_retriever()

In [137]:
# Retrieval Grader
# Data model
class GradeDocuments(BaseModel):
    """Binary score for relevance check on retrieved documents."""

    binary_score: str = Field(
        description="Documents are relevant to the question, 'yes' or 'no'"
    )


# LLM with function call
structured_llm_grader = llm.with_structured_output(GradeDocuments)

# Prompt
system = """You are a grader assessing relevance of a retrieved document to a user question. \n 
    It does not need to be a stringent test. The goal is to filter out erroneous retrievals. \n
    If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."""
grade_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
    ]
)

retrieval_grader = grade_prompt | structured_llm_grader
question = "agent memory"
docs = retriever.get_relevant_documents(question)
# doc_txt = docs[1].page_content
# print(retrieval_grader.invoke({"question": question, "document": doc_txt}))

final_docs = []
for doc in docs:
    binary_result = retrieval_grader.invoke({"question": question, "document": doc.metadata['description']})
    if binary_result.binary_score == 'yes':
        final_docs.append(doc.metadata['description'])

binary_score='no'


In [209]:
### Generate

from langchain import hub
from langchain_core.output_parsers import StrOutputParser

# Prompt
prompt = hub.pull("rlm/rag-prompt")

# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# Chain
rag_chain = prompt | llm | StrOutputParser()

# Run
generation = rag_chain.invoke({"context": final_docs, "question": question})
print(generation)

C:\Users\white\anaconda3\envs\env_rag\lib\site-packages\langsmith\client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(
Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised InternalServerError: 500 An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting.


Agent memory consists of two types: short-term and long-term. Short-term memory utilizes in-context learning for immediate learning. Long-term memory allows the agent to retain and recall information over extended periods, often using external vector stores.


In [210]:
### Hallucination Grader

# Data model
class GradeHallucinations(BaseModel):
    """Binary score for hallucination present in generation answer."""

    binary_score: str = Field(
        description="Answer is grounded in the facts, 'yes' or 'no'"
    )

# LLM with function call
structured_llm_grader = llm.with_structured_output(GradeHallucinations)

# Prompt
system = """You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts. \n 
     Give a binary score 'yes' or 'no'. 'Yes' means that the answer is grounded in / supported by the set of facts."""
hallucination_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "Set of facts: \n\n {documents} \n\n LLM generation: {generation}"),
    ]
)

hallucination_grader = hallucination_prompt | structured_llm_grader
hallucination_grader.invoke({"documents": final_docs, "generation": generation})

GradeHallucinations(binary_score='yes')

In [211]:
### Answer Grader


# Data model
class GradeAnswer(BaseModel):
    """Binary score to assess answer addresses question."""

    binary_score: str = Field(
        description="Answer addresses the question, 'yes' or 'no'"
    )


# LLM with function call
structured_llm_grader = llm.with_structured_output(GradeAnswer)

# Prompt
system = """You are a grader assessing whether an answer addresses / resolves a question \n 
     Give a binary score 'yes' or 'no'. Yes' means that the answer resolves the question."""
answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "User question: \n\n {question} \n\n LLM generation: {generation}"),
    ]
)

answer_grader = answer_prompt | structured_llm_grader
answer_grader.invoke({"question": question, "generation": generation})

GradeAnswer(binary_score='yes')

In [212]:
### Question Re-writer

# Prompt
system = """You a question re-writer that converts an input question to a better version that is optimized \n 
     for vectorstore retrieval. Look at the input and try to reason about the underlying semantic intent / meaning."""
re_write_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        (
            "human",
            "Here is the initial question: \n\n {question} \n Formulate an improved question.",
        ),
    ]
)

question_rewriter = re_write_prompt | llm | StrOutputParser()
question_rewriter.invoke({"question": question})


'What are the different types of agent memory and how do they function?'